# P55 again — exposure-bias rollout seeded with several real frames (Wan2.2, ball_drop_s252)

Only the P55 part of `p43_59_3.ipynb`, with one change: instead of giving the model a single real latent frame (the empty
room) and generating everything else, the rollout is **seeded with the first `N_SEED` real latent frames**, so the ball is
already falling when the model takes over. After the seeds it runs **free**: every new frame is generated from the model's
own earlier output.

Put this file in the Wan2.2 repo root (next to `p43_59_w22.py`) and run it from there. Nothing from the earlier notebook
needs re-running; it reads the saved latents of the run. Outputs go to `attn_runs/ball_drop_s252/p43_59_3/p55/` under new
names (`seedN_MODE_free`), so the existing rollouts are left alone.

**Time on the 3090**, from the pace of the earlier rollouts: with `N_SEED = 5` there are 8 frames to generate, about
**50 minutes** per rollout. Progress is saved after every generated frame, so an interrupted run continues where it stopped.

**What to expect.** The earlier teacher-forced rollout also fell apart, even though it saw the real frames, so the patched
one-frame-at-a-time model does not use its context well. The seeds guarantee the first `N_SEED` frames show the real falling
ball; how many frames the bounce survives after that is what this run finds out.

## 0. Setup

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

REPO = Path.cwd()                                   # run from the Wan2.2 repo root
sys.path.insert(0, str(REPO))
import wan22_runtime as w22
import wan22_capture as wc
import p43_59_w22 as pw

BLOCKS_DIR = Path('~/data/Wan2.2-T2V-A14B-blocks').expanduser()
AUX_DIR    = Path('~/data/Wan2.1-T2V-1.3B').expanduser()       # VAE
RUN_DIR    = REPO / 'attn_runs' / 'ball_drop_s252'
WINDOW     = range(1, 13)                           # latent frames of the full clip to keep
DEVICE     = 'cuda:0' if torch.cuda.is_available() else 'cpu'

rt  = w22.Runtime(BLOCKS_DIR, AUX_DIR, device=DEVICE)     # experts and VAE are loaded on demand
run = wc.WindowRun(RUN_DIR, keep_frames=WINDOW)
OUT = RUN_DIR / 'p43_59_3'
S   = pw.Session(rt, run, OUT)
P55 = S.sub('p55')
print('prompt:', run.meta['prompt'])
print(f'window: {run.F} frames, {S.n_video} video frames | {S.n_steps} steps | out -> {P55}')

## 1. Settings

Latent frame `f` of the clip is frame `f - 1` of the 12-frame window (latent frame 0, the empty room, is outside the window
and never shown). In the original clip the ball enters at window frame 1, falls through frames 2-3 and is on its way back up
by frame 4. `N_SEED = 5` gives the model latent frames 0-4 = window frames 0-3: the whole first fall.

| `N_SEED` | real window frames | first generated window frame | frames to generate | rough time |
|---|---|---|---|---|
| 4 | 0-2 | 3 | 9 | ~55 min |
| 5 | 0-3 | 4 | 8 | ~50 min |
| 6 | 0-4 | 5 | 7 | ~46 min |

In [ ]:
N_SEED        = 5                        # real latent frames given before generation starts (latent frames 0 .. N_SEED-1)
CONTEXT_MODES = ['clean_t0']             # one rollout per entry; also available: 'boundary', 'renoise'
RUN_TEACHER   = False                    # also run the teacher-forced version of each (same cost again)
RUN_PILOT     = False                    # section 2: two generated frames per context mode first (~9 min per mode)

P55_STEPS    = 20                        # denoising steps per generated frame
P55_CHUNK    = 1                         # latent frames generated per AR step
CAUSAL_ATTN  = True
P55_FRAMES   = run.keep[-1] + 1          # latent frames on the clip's timeline: 0 .. 12
DROP         = [f for f in range(P55_FRAMES) if f not in run.keep]     # generated/given but not shown: [0]

gt     = S.latent('final')[:, :P55_FRAMES]          # the finished clip's latents
noise0 = S.latent('step_00')[:, :P55_FRAMES]        # the original run's initial noise
ref    = run.video                                   # the window of the original clip (48 video frames)
assert 1 <= N_SEED < P55_FRAMES
first_gen = N_SEED - len([f for f in DROP if f < N_SEED])      # first generated frame, in window numbering
print(f'{P55_FRAMES} latent frames: {N_SEED} given (window frames 0..{first_gen - 1} are real), '
      f'{P55_FRAMES - N_SEED} generated (window frames {first_gen}..{run.F - 1}), {P55_STEPS} steps each')
print(f'video frames 0..{4 * first_gen - 1} of each rollout are real, {4 * first_gen}..{S.n_video - 1} generated')

## 2. Optional pilot: which context mode?

Off by default (`RUN_PILOT`). Free-running, first two generated frames only, once per mode, so you can see in about half an
hour whether any mode keeps the ball moving before spending 50 minutes on a full rollout. Then put the mode(s) you want in
`CONTEXT_MODES` above.

In [ ]:
if RUN_PILOT:
    PILOT_MODES  = ['clean_t0', 'boundary', 'renoise']
    PILOT_FRAMES = N_SEED + 2
    pilot = {}
    for mode in PILOT_MODES:
        name = f'pilot_seed{N_SEED}_{mode}_free'
        if (P55 / name / 'metrics.json').exists():
            pilot[mode] = np.load(P55 / name / 'video.npy'); continue
        print(name)
        lat = pw.ar_rollout(S, total_frames=PILOT_FRAMES, gt=gt, noise0=noise0, n_seed=N_SEED, teacher_forced=False,
                            chunk=P55_CHUNK, context_mode=mode, causal=CAUSAL_ATTN, steps=P55_STEPS)['latents']
        pilot[mode] = pw.export_rollout(S, name, lat, P55, extra=dict(context_mode=mode, steps=P55_STEPS, n_seed=N_SEED),
                                        drop_latent_frames=DROP)['video']
    n = min(len(v) for v in pilot.values())
    cols = list(range(max(0, n - 12), n))                      # the last seeded frame + the 8 generated video frames
    rows = [('original', ref)] + list(pilot.items())
    fig, axes = plt.subplots(len(rows), len(cols), figsize=(len(cols) * 2.6, len(rows) * 1.62 + 0.4), squeeze=False)
    for r, (nm, v) in enumerate(rows):
        for c, f in enumerate(cols):
            axes[r, c].imshow(v[f][::2, ::2]); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(f'video frame {f}' + ('  (real)' if f < 4 * first_gen else ''), fontsize=8)
        axes[r, 0].set_ylabel(nm, fontsize=9)
    fig.suptitle(f'pilot: free-running after {N_SEED} real latent frames, {P55_STEPS} steps', fontsize=11)
    fig.tight_layout(); fig.savefig(P55 / f'pilot_seed{N_SEED}.png', dpi=130); plt.close(fig)
    display(IPImage(str(P55 / f'pilot_seed{N_SEED}.png'), width=1500))
else:
    print('pilot skipped (RUN_PILOT = False)')

## 3. The rollout(s)

Per rollout in `p55/NAME/`: `latents.pt`, `video.npy`, `video.mp4`, `frames/frame_000..047.png`, `strip.png`,
`metrics.json`. The 48 frames line up with the window (4 per latent frame), exactly like the earlier rollouts and the
`p43/xt/step_40_final` frames.

In [ ]:
def run_or_load(name, teacher_forced, mode):
    d = P55 / name
    if (d / 'metrics.json').exists():
        print(name, ': already on disk')
        return dict(dir=d, video=np.load(d / 'video.npy'), metrics=json.loads((d / 'metrics.json').read_text()))
    print(name)
    cfg = dict(total_frames=P55_FRAMES, n_seed=N_SEED, teacher_forced=teacher_forced, chunk=P55_CHUNK,
               context_mode=mode, causal=CAUSAL_ATTN, steps=P55_STEPS)
    lat = pw.ar_rollout(S, gt=gt, noise0=noise0, resume=P55 / f'{name}_progress.pt', **cfg)['latents']
    return pw.export_rollout(S, name, lat, P55, ref_video=ref, extra=cfg, drop_latent_frames=DROP)

rollouts = {}
for mode in CONTEXT_MODES:
    for tf in ([False, True] if RUN_TEACHER else [False]):
        kind = 'teacher' if tf else 'free'
        rollouts[f'{mode} {kind}'] = run_or_load(f'seed{N_SEED}_{mode}_{kind}', tf, mode)
for k, v in rollouts.items():
    print(f'{k:20s} -> {v["dir"] / "frames"}')

## 4. Look at it

In [ ]:
rows = {'original (bidirectional)': ref, **{k: v['video'] for k, v in rollouts.items()}}
tag = f'seed{N_SEED}'
p = pw.compare_rollouts(S, rows, P55 / f'compare_{tag}.png',
                        title=f'rollout after {N_SEED} real latent frames (window frames 0..{first_gen - 1} real), '
                              f'{P55_STEPS} steps per frame')
display(IPImage(str(p), width=1500))
p = pw.plot_drift({k: v['metrics'] for k, v in rollouts.items()}, P55 / f'drift_{tag}.png', title=f'rollout, {tag}')
display(IPImage(str(p), width=1300))

In [ ]:
# every generated video frame of each rollout (plus the last real one), full detail, for picking stills
for k, v in rollouts.items():
    vid = v['video']
    cols = list(range(4 * first_gen - 1, len(vid)))
    per_row = 11
    n_rows = -(-len(cols) // per_row)
    fig, axes = plt.subplots(n_rows, per_row, figsize=(per_row * 2.6, n_rows * 1.72), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for i, f in enumerate(cols):
        ax = axes[i // per_row, i % per_row]
        ax.imshow(vid[f][::2, ::2]); ax.set_title(f'frame_{f:03d}' + ('  (real)' if f < 4 * first_gen else ''), fontsize=8)
    fig.suptitle(k, fontsize=11); fig.tight_layout()
    path = v['dir'] / 'generated_frames.png'
    fig.savefig(path, dpi=110); plt.close(fig)
    display(IPImage(str(path), width=1500))

## Notes for the manim side

* Frames are in `p43_59_3/p55/seedN_MODE_free/frames/frame_000 … frame_047.png`, numbered like the window: latent frame `j`
  of the window is video frames `4j … 4j+3`.
* The first `4 * (N_SEED - 1)` video frames are the real clip (decoded from the real latents inside the rollout, so they can
  differ from `p43/xt/step_40_final` by a hair at the hand-over); everything after is the model running on its own output.
* `metrics.json` has `config` (the settings) and per-frame PSNR against the original.